# UTS Praktikum Penelusuran Informasi
## Multi-Dataset Information Retrieval — versi notebook

**Kelompok:** [isi]  
**Anggota / NPM:** [isi maksimal 3 anggota]

Notebook ini mengadaptasi proyek CLI sebelumnya. Semua fungsi inti berada di notebook ini sehingga tidak perlu mengimpor file Python proyek. Dataset tetap dibaca dari file CSV, tidak ditanam ke dalam notebook.

**Cara pakai:** letakkan notebook di folder proyek hasil ekstrak ZIP, sejajar dengan `datasets/`, lalu pilih kernel Python dan jalankan sel berurutan dari atas. Alternatif: letakkan `dataset(1).zip` atau `dataset.zip` di folder kerja; bagian B akan menyiapkan kelima folder. Indexing penuh dapat memerlukan beberapa menit dan ruang disk beberapa GB.

Menu CLI untuk demonstrasi tugas tersedia di bagian H. `Run All` tidak membuka input interaktif secara otomatis. Pengumpulan tugas tetap membutuhkan laporan PDF dan README dari paket sebelumnya.

## A. Persiapan
### 1. Instalasi dependensi
Jalankan sekali pada kernel yang dipilih. `%pip` memasang paket pada lingkungan notebook. Internet diperlukan jika dependensi belum terpasang.

In [ ]:
%pip install "Whoosh==2.7.4" "scikit-learn>=1.3,<2" "pandas>=1.5,<3"

### 2. Import library
Library utama: Whoosh untuk index dan pencarian, CountVectorizer untuk Bag of Words, serta cosine similarity untuk ranking. Pandas hanya untuk tampilan tabel.

In [ ]:
import csv
import json
import shutil
import tempfile
from collections import Counter
from pathlib import Path
from time import perf_counter

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from whoosh import index
from whoosh.analysis import SpaceSeparatedTokenizer
from whoosh.fields import Schema, ID, TEXT, NUMERIC
from whoosh.query import Or, Term


import pandas as pd


def display(table):
    """Tabel teks agar output tetap berbasis CLI."""
    print(table.to_string() if isinstance(table, pd.DataFrame) else table)


## B. Dataset
### 3. Tentukan lokasi dan siapkan lima sumber
`Path.cwd()` adalah folder kerja kernel. Ubah `BASE_DIR` bila folder kerja berbeda. Bila folder `datasets/` sudah tersedia, ZIP tidak diperlukan. CSV harus memiliki kolom `judul` dan `konten`.

In [ ]:
import zipfile

BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / 'datasets'
INDEX_DIR = BASE_DIR / 'indexdir'
SOURCES = ('etd-usk', 'etd-ugm', 'kompas', 'tempo', 'mojok')

missing = [s for s in SOURCES if not list((DATASET_DIR / s).glob('*.csv'))]
if missing:
    archives = [BASE_DIR / 'dataset(1).zip', BASE_DIR / 'dataset.zip']
    archive = next((p for p in archives if p.is_file()), None)
    if archive is None:
        raise FileNotFoundError(
            f'Folder kerja: {BASE_DIR}. Letakkan datasets/ di folder ini '
            'atau ubah BASE_DIR. Alternatif: sediakan dataset(1).zip atau dataset.zip.'
        )
    # Salin hanya lima CSV yang dikenali; tidak mengekstrak path sembarang.
    with zipfile.ZipFile(archive) as zipped:
        for source in missing:
            expected = source.replace('-', '_') + '.csv'
            matches = [n for n in zipped.namelist() if Path(n).name == expected]
            if len(matches) != 1:
                raise ValueError(f'CSV {expected} tidak ditemukan atau ambigu dalam ZIP.')
            target = DATASET_DIR / source / 'data.csv'
            target.parent.mkdir(parents=True, exist_ok=True)
            with zipped.open(matches[0]) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst)

print('Folder dataset:', DATASET_DIR)
print('Sumber siap:', ', '.join(SOURCES))

## C. Preprocessing
### 4. Case folding, tokenisasi, dan stopword removal
Normalisasi Unicode, decoding HTML, penghapusan tag, lowercase, pemisahan token alfanumerik, dan penghapusan stopword diterapkan sama pada dokumen serta query. Tidak menggunakan stemming.

In [ ]:
"""Preprocessing yang sama untuk dokumen dan query, tanpa unduhan eksternal."""
import html
import re
import unicodedata

# Daftar eksplisit dan dapat diperluas; negasi 'tidak' sengaja dipertahankan.
STOPWORDS = frozenset('''yang dan di ke dari untuk pada dengan adalah ini itu dalam
atau sebagai oleh akan telah sudah juga karena agar sehingga saat ketika antara
para sebuah suatu tersebut terhadap tentang kepada saya kamu kami kita mereka ia
dia nya pun lah kah bahwa yaitu yakni serta bisa dapat lebih paling sangat ada
itu ini the a an and or of to in on for with is are was were be been this that
as by from at it its'''.split())

def tokenize(text):
    text = unicodedata.normalize('NFKC', html.unescape(str(text))).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    return [t for t in re.findall(r'[^\W_]+', text, flags=re.UNICODE)
            if t not in STOPWORDS]

def preprocess(text):
    return ' '.join(tokenize(text))


### 5. Contoh hasil preprocessing

In [ ]:
contoh = ['<b>PENDIDIKAN</b> dan Matematika!', 'Gempa di Aceh pada 2025', 'dan yang di']
display(pd.DataFrame({'Teks asli': contoh, 'Hasil preprocessing': [preprocess(t) for t in contoh]}))

## D. Mesin IR
### 6. Bag of Words, Whoosh, dan cosine similarity
Kelas `SearchEngine` mempertahankan algoritma proyek sebelumnya:

- `build()` membaca CSV, membentuk BoW sparse dengan CountVectorizer, dan membuat index Whoosh.
- `load()` membuka index tersimpan dan membangun ulang BoW dari teks bersih.
- `search()` mengambil semua kandidat yang memiliki minimal satu term query dengan Whoosh OR, menghitung cosine, lalu mengembalikan top-5.

Cosine = dot product kedua vektor dibagi hasil kali panjang vektornya. Skor bukan persentase akurasi. Query di luar vocabulary atau hanya stopword tidak menghasilkan dokumen. Skor bawaan Whoosh tidak digunakan sebagai ranking final.

In [ ]:
SOURCES = ('etd-usk', 'etd-ugm', 'kompas', 'tempo', 'mojok')

class SearchEngine:
    def __init__(self, index_dir):
        self.index_dir = Path(index_dir)
        self.ix = None
        self.vectorizer = None
        self.matrix = None
        self.docs = []

    def build(self, dataset_dir):
        """Validasi semua sumber sebelum membangun index pengganti."""
        start = perf_counter()
        root = Path(dataset_dir)
        docs, counts = [], Counter()
        skipped = 0
        for source in SOURCES:
            files = sorted((root / source).glob('*.csv'))
            if not files:
                raise ValueError(f'Dataset {source} tidak ditemukan di {root}.')
            for file in files:
                with file.open(encoding='utf-8-sig', newline='') as stream:
                    reader = csv.DictReader(stream)
                    if not {'judul', 'konten'} <= set(reader.fieldnames or []):
                        raise ValueError(f'{file}: wajib memiliki kolom judul dan konten.')
                    for row_no, row in enumerate(reader, 2):
                        title = (row.get('judul') or '').strip()
                        content = (row.get('konten') or '').strip()
                        clean = preprocess(title + ' ' + content)
                        if not clean:
                            skipped += 1
                            continue
                        docs.append(dict(doc_id=f'{source}/{file.name}:{row_no}',
                                         source=source, title=title or '(Tanpa judul)',
                                         content=content, clean=clean, row_id=len(docs)))
                        counts[source] += 1
            if not counts[source]:
                raise ValueError(f'Dataset {source} tidak memiliki dokumen valid.')
        vectorizer = CountVectorizer(tokenizer=str.split, token_pattern=None,
                                     preprocessor=None, lowercase=False)
        matrix = vectorizer.fit_transform(d['clean'] for d in docs)
        schema = Schema(doc_id=ID(stored=True, unique=True), source=ID(stored=True),
                        title=TEXT(stored=True), content=TEXT(stored=True),
                        clean=TEXT(stored=True, analyzer=SpaceSeparatedTokenizer()),
                        row_id=NUMERIC(stored=True))
        self.index_dir.parent.mkdir(parents=True, exist_ok=True)
        temp = Path(tempfile.mkdtemp(prefix='ir_build_', dir=self.index_dir.parent))
        try:
            ix = index.create_in(temp, schema)
            writer = ix.writer()
            for doc in docs:
                writer.add_document(**doc)
            writer.commit()
            ix.close()
            if self.ix is not None:
                self.ix.close()
            if self.index_dir.exists():
                shutil.rmtree(self.index_dir)
            temp.rename(self.index_dir)
        finally:
            if temp.exists():
                shutil.rmtree(temp)
        self.ix = index.open_dir(self.index_dir)
        self.docs, self.vectorizer, self.matrix = docs, vectorizer, matrix
        return dict(total=len(docs), per_source=dict(counts), skipped=skipped,
                    vocabulary=len(vectorizer.vocabulary_),
                    nonzero=int(matrix.nnz), seconds=perf_counter()-start)

    def load(self):
        """Rekonstruksi BoW dari teks bersih tersimpan, tanpa pickle."""
        self.ix = index.open_dir(self.index_dir)
        with self.ix.searcher() as searcher:
            self.docs = sorted(searcher.all_stored_fields(), key=lambda d:d['row_id'])
        self.vectorizer = CountVectorizer(tokenizer=str.split, token_pattern=None,
                                         preprocessor=None, lowercase=False)
        self.matrix = self.vectorizer.fit_transform(d['clean'] for d in self.docs)

    def search(self, query, top_k=5):
        if self.ix is None:
            raise ValueError('Index belum tersedia. Pilih menu 1 terlebih dahulu.')
        terms = preprocess(query).split()
        if not terms:
            return []
        vector = self.vectorizer.transform([' '.join(terms)])
        if vector.nnz == 0:
            return []
        # OR tanpa limit: semua dokumen dengan overlap harus ikut cosine.
        query_object = Or([Term('clean', t) for t in sorted(set(terms))])
        with self.ix.searcher() as searcher:
            candidates = [hit['row_id'] for hit in searcher.search(query_object, limit=None)]
        if not candidates:
            return []
        scores = cosine_similarity(vector, self.matrix[candidates]).ravel()
        ranked = sorted(zip(candidates, scores),
                        key=lambda pair: (-float(pair[1]), self.docs[pair[0]]['doc_id']))
        return [dict(rank=rank, doc_id=self.docs[i]['doc_id'],
                     source=self.docs[i]['source'], title=self.docs[i]['title'],
                     score=float(score), snippet=self.docs[i]['content'][:240])
                for rank, (i, score) in enumerate(ranked[:top_k], 1) if score > 0]


### 7. Bangun atau muat index
Index pertama kali dibangun dari kelima sumber lengkap. Jika index sudah ada, sel memuatnya kembali. Ubah `REBUILD = True` setelah mengubah CSV atau aturan preprocessing. Jalankan sel ini sebelum mencari. Jangan hentikan kernel saat index sedang ditulis.

In [ ]:
REBUILD = False
engine = SearchEngine(INDEX_DIR)
start = perf_counter()
if not REBUILD and index.exists_in(str(INDEX_DIR)):
    print('Memuat index dan membangun ulang BoW...')
    engine.load()
    stats = dict(total=len(engine.docs), per_source=dict(Counter(d['source'] for d in engine.docs)),
                 vocabulary=len(engine.vectorizer.vocabulary_), nonzero=int(engine.matrix.nnz))
else:
    print('Membangun index seluruh dataset. Proses ini dapat memerlukan beberapa menit...')
    stats = engine.build(DATASET_DIR)
print(f'Siap. Waktu sel: {perf_counter() - start:.2f} detik')
display(pd.DataFrame(list(stats['per_source'].items()), columns=['Sumber', 'Dokumen']))
display(pd.DataFrame([{'Total dokumen': stats['total'], 'Vocabulary': stats['vocabulary'],
                      'Elemen tidak nol': stats['nonzero']}]))

## E. Pencarian
### 8. Fungsi tampilan hasil
Tabel menampilkan peringkat, sumber, judul, skor, ID, dan cuplikan. Urutan skor yang sama ditentukan oleh ID dokumen.

In [ ]:
def tampilkan_hasil(results):
    if not results:
        print('Tidak ditemukan hasil. Query mungkin kosong, hanya stopword, atau di luar vocabulary.')
        return
    frame = pd.DataFrame(results)[['rank', 'source', 'title', 'score', 'doc_id', 'snippet']]
    with pd.option_context('display.max_colwidth', 120, 'display.float_format', '{:.6f}'.format):
        display(frame.set_index('rank'))


### 9. Cari query sendiri
Ubah teks `QUERY`, lalu jalankan ulang sel ini saja untuk pencarian berikutnya.

In [ ]:
QUERY = 'pendidikan matematika'
start = perf_counter()
hasil = engine.search(QUERY)
print('Query:', QUERY)
print(f'Waktu pencarian: {perf_counter() - start:.4f} detik')
tampilkan_hasil(hasil)

## F. Pengujian
### 10. Lima query topik dan tiga kasus batas
Hasil dibuat saat sel dijalankan, bukan angka yang diketik manual. Dataset tanpa label relevansi belum memungkinkan perhitungan precision/recall yang bermakna.

In [ ]:
QUERIES = ['pendidikan matematika', 'gempa Aceh', 'kecerdasan buatan',
           'korupsi pemerintah', 'lagu anime', '', 'dan yang di', 'zzzxxyyqq']
pengujian = []
for query in QUERIES:
    start = perf_counter()
    results = engine.search(query)
    pengujian.append(dict(query=query, seconds=perf_counter()-start, results=results))
display(pd.DataFrame([{'Query': repr(t['query']), 'Jumlah hasil': len(t['results']),
                       'Waktu (detik)': t['seconds'],
                       'Skor teratas': t['results'][0]['score'] if t['results'] else None}
                      for t in pengujian]))
for test in pengujian[:5]:
    print('\nQuery:', test['query'])
    tampilkan_hasil(test['results'])

### 11. Verifikasi cosine dan input batas
Contoh query `apel jeruk` = (1,1), dokumen `apel apel jeruk` = (2,1). Cosine yang diharapkan adalah 3 / akar(10).

In [ ]:
mini = CountVectorizer()
mini_matrix = mini.fit_transform(['apel apel jeruk', 'pisang'])
actual = cosine_similarity(mini.transform(['apel jeruk']), mini_matrix)[0, 0]
expected = 3 / (10 ** 0.5)
assert abs(actual - expected) < 1e-12
assert preprocess('<b>PENDIDIKAN</b> dan Matematika!') == 'pendidikan matematika'
assert engine.search('') == []
assert engine.search('dan yang di') == []
print(f'Cosine aktual = {actual:.6f}; ekspektasi = {expected:.6f}')
print('Pemeriksaan lulus.')

## G. Simpan Hasil
### 12. Ekspor hasil ke JSON
Berkas dapat dipakai sebagai bukti untuk laporan. Waktu akan berbeda pada setiap perangkat.

In [ ]:
OUTPUT_DIR = BASE_DIR / 'hasil_pengujian_notebook'
OUTPUT_DIR.mkdir(exist_ok=True)
output_file = OUTPUT_DIR / 'results.json'
output_file.write_text(json.dumps({'stats': stats, 'tests': pengujian},
                                 ensure_ascii=False, indent=2), encoding='utf-8')
print('Hasil disimpan:', output_file)

## H. Menu CLI Interaktif Sesuai Tugas
### 13. Menu sesuai tugas
Ubah `JALANKAN_MENU = True`, kemudian jalankan sel. Menu 1 membangun ulang index, menu 2 meminta query, dan menu 3 keluar. Gunakan bagian E untuk pencarian tanpa input interaktif.

In [ ]:
def menu_cli():
    while True:
        print('\n=== INFORMATION RETRIEVAL SYSTEM ===')
        print('[1] Load & Index Dataset\n[2] Search Query\n[3] Exit')
        try:
            pilihan = input('Pilihan: ').strip()
            if pilihan == '1':
                print('Membangun ulang index...')
                print(engine.build(DATASET_DIR))
            elif pilihan == '2':
                tampilkan_hasil(engine.search(input('Query: ')))
            elif pilihan == '3':
                print('Selesai.')
                break
            else:
                print('Pilih 1, 2, atau 3.')
        except (EOFError, KeyboardInterrupt):
            print('\nMenu dihentikan.')
            break
        except (ValueError, OSError) as error:
            print('Gagal:', error)

JALANKAN_MENU = False
if JALANKAN_MENU:
    menu_cli()
else:
    print('Menu tidak dijalankan. Ubah JALANKAN_MENU menjadi True jika diperlukan.')

## I. Kesimpulan dan Catatan
Pipeline menggabungkan lima sumber, preprocessing, representasi BoW, Whoosh, dan cosine similarity. Judul serta isi mendapat bobot frekuensi yang sama. BoW tidak memahami sinonim atau konteks: kata yang sama dapat berasal dari topik berbeda. Semua kandidat Whoosh OR dihitung sebelum diambil top-5.

**Jika bermasalah:**
- `ModuleNotFoundError`: jalankan sel instalasi pada kernel yang sama.
- Dataset tidak ditemukan: periksa `BASE_DIR` pada bagian B.
- `NameError`: jalankan sel sebelumnya sesuai urutan.
- Indexing lama: tunggu sel 7 selesai; gunakan kembali index pada sesi berikutnya.
- Setelah mengganti preprocessing/dataset: set `REBUILD = True` untuk satu kali pembangunan ulang.

Notebook dibagikan tanpa output tersimpan agar tidak menampilkan hasil lama sebagai hasil eksekusi baru. Jalankan sel untuk mendapatkan tabel dan skor pada perangkatmu.